In [ ]:
# Hardcoded version for Jupyter Notebook
# out_dir を削除し、図は表示のみ

from pathlib import Path
from typing import Dict, List, Tuple
import h5py
import matplotlib.pyplot as plt
import pandas as pd

from ppo_dual_xarm7 import CONTROL_FREQUENCY_HZ  # use control freq to convert steps→time

# =====================
# Hardcoded paths
# =====================
csv_root = Path("runs/MyDualBoxRotationAblated-v0__ppo_dual_xarm7__1__1763351759/info/rollout")
h5_path = Path("/home/kotaro/my_projects/robot/RoboManipBaselines/robo_manip_baselines/dataset/RolloutPpoCus_RealXarm7DualDemo_20251119_142942/RealXarm7DualDemo_world0_000.rmb/main.rmb.hdf5")
env_name = None  # e.g. "env_000" or None for auto

# =====================
# Helper functions
# =====================
def load_env_csvs(root: Path, subdir: str) -> Dict[str, pd.DataFrame]:
    folder = root / subdir
    if not folder.exists():
        raise FileNotFoundError(f"Directory not found: {folder}")
    data: Dict[str, pd.DataFrame] = {}
    for csv_path in sorted(folder.glob("env_*.csv")):
        df = pd.read_csv(csv_path)
        if "step" not in df.columns:
            raise ValueError(f"Missing 'step' column in {csv_path}")
        data[csv_path.stem] = df.set_index("step")
    if not data:
        raise FileNotFoundError(f"No env_*.csv files found in {folder}")
    return data


def load_real_h5(h5_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    with h5py.File(h5_path, "r") as f:
        time = pd.Series(f["time"][:], name="time")
        measured = pd.DataFrame(f["measured_joint_pos"][:])
        command = pd.DataFrame(f["command_joint_pos"][:])
    return measured, command, time


def plot_joint(
    joint_idx: int,
    sim_time,
    sim_measured,
    sim_command,
    real_time,
    real_measured,
    real_command,
    env_name: str,
) -> None:
    plt.figure(figsize=(8, 4))
    plt.plot(sim_time, sim_measured, label="sim measured q", linewidth=1.5)
    plt.plot(sim_time, sim_command, label="sim command q", linestyle="--", linewidth=1.5)
    plt.plot(real_time, real_measured, label="real measured q", linewidth=1.5)
    plt.plot(real_time, real_command, label="real command q", linestyle="--", linewidth=1.5)
    plt.title(f"{env_name} - Joint {joint_idx}")
    plt.xlabel("time (s)")
    plt.ylabel("joint position")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# =====================
# Main logic for notebook
# =====================

sim_measured = load_env_csvs(csv_root, "mesured_q")
sim_direct_joint_command = load_env_csvs(csv_root, "direct_joint_command")
common_envs = sorted(set(sim_measured) & set(sim_direct_joint_command))
if not common_envs:
    raise RuntimeError("No common env_*.csv between mesured_q and direct_joint_command.")

if env_name is None:
    env_name = common_envs[0]
if env_name not in common_envs:
    raise ValueError(f"Env {env_name} not found; available: {common_envs}")

real_measured, real_command, real_time = load_real_h5(h5_path)

sim_measured_df = sim_measured[env_name]
sim_direct_joint_command_df = sim_direct_joint_command[env_name]
sim_steps = sim_measured_df.index.to_numpy()
sim_time = sim_steps / CONTROL_FREQUENCY_HZ  # convert step index to seconds

joints: List[int] = sorted(
    int(c.split("_")[-1]) for c in sim_measured_df.columns if c.startswith("mesured_q_")
)

for j in joints:
    m_col = f"mesured_q_{j}"
    a_col = f"direct_joint_command_{j}"
    if a_col not in sim_direct_joint_command_df.columns:
        print(f"Skip joint {j}: {a_col} not in direct_joint_command")
        continue
    if j >= real_measured.shape[1] or j >= real_command.shape[1]:
        print(f"Skip joint {j}: not present in real data")
        continue
    plot_joint(
        j,
        sim_time,
        sim_measured_df[m_col].to_numpy(),
        sim_direct_joint_command_df[a_col].to_numpy(),
        real_time.to_numpy(),
        real_measured.iloc[:, j].to_numpy(),
        real_command.iloc[:, j].to_numpy(),
        env_name,
    )


In [ ]:
print("sim_steps:", sim_steps.shape)
print("sim_measured:", sim_measured_df[m_col].shape)
print("sim_command:", sim_direct_joint_command_df[a_col].shape)
